# Milestone 1: Data Profiling

Notebook ini bertujuan untuk melakukan inspeksi dan *profiling* awal terhadap dataset **Wikipedia Clickstream** (berukuran jutaan baris) secara efisien menggunakan **Polars**.

In [7]:
import polars as pl
import glob

file_list = glob.glob("../data/raw/*.tsv.gz")

lazy_frames = []
for f in file_list:
    lf = pl.scan_csv(
        f, 
        separator='\t', 
        has_header=False,
        quote_char=None,
        ignore_errors=True,
        new_columns=["halaman_asal", "halaman_tujuan", "tipe_link", "jumlah_klik"]
    )
    lazy_frames.append(lf)

df = pl.concat(lazy_frames)

print(f"Berhasil memuat {len(file_list)} file .gz dalam bentuk LazyFrame!")

Berhasil memuat 4 file .gz dalam bentuk LazyFrame!


### 1. Informasi Baris dan Kolom Dasar

In [8]:
total_rows = df.select(pl.len()).collect().item()
print(f"Total baris data: {total_rows:,}")

print("\nSkema Kolom & Tipe Data:")
print(df.schema)

Total baris data: 2,475,926

Skema Kolom & Tipe Data:
Schema({'halaman_asal': String, 'halaman_tujuan': String, 'tipe_link': String, 'jumlah_klik': Int64})


/tmp/ipykernel_361/822951732.py:5: PerformanceWarning: Resolving the schema of a LazyFrame is a potentially expensive operation. Use `LazyFrame.collect_schema()` to get the schema without this warning.
  print(df.schema)


### 2. Mengintip Sampel Data (Head)

In [9]:
print("5 Baris Pertama:")
display(df.head(5).collect())

5 Baris Pertama:


halaman_asal,halaman_tujuan,tipe_link,jumlah_klik
str,str,str,i64
"""other-empty""","""Halaman_Utama""","""external""",683510
"""other-empty""","""Kleopatra""","""external""",238768
"""other-internal""","""Partai_Kasih""","""external""",195370
"""other-search""","""Bahlil_Lahadalia""","""external""",155770
"""other-empty""","""DuckDuckGo""","""external""",138430


### 3. Pengecekan Nilai Kosong (Null Values)

In [10]:
null_counts = df.select([
    pl.col(c).null_count().alias(c) for c in df.columns
]).collect()

print("Jumlah Nilai Kosong per Kolom:")
display(null_counts)

/tmp/ipykernel_361/1107593460.py:2: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  pl.col(c).null_count().alias(c) for c in df.columns


Jumlah Nilai Kosong per Kolom:


halaman_asal,halaman_tujuan,tipe_link,jumlah_klik
u32,u32,u32,u32
0,0,0,0


### 4. Statistik Deskriptif Dasar (Angka)

In [11]:
stats = df.select("jumlah_klik").collect().describe()
display(stats)

statistic,jumlah_klik
str,f64
"""count""",2.475926e6
"""null_count""",0.0
"""mean""",97.577048
"""std""",2428.588548
"""min""",10.0
"""25%""",14.0
"""50%""",22.0
"""75%""",50.0
"""max""",3.055641e6


### 5. Profiling Kategori: Persebaran Tipe Link
Melihat jenis sumber lalu lintas (external vs link internal).

In [12]:
link_types = (
    df.group_by("tipe_link")
    .agg(pl.len().alias("frekuensi"))
    .sort("frekuensi", descending=True)
).collect()

display(link_types)

tipe_link,frekuensi
str,u32
"""external""",1497179
"""link""",938974
"""other""",39773
